# EXPERIMENT 9
## Transformer-Based LLM Chatbot Using Streamlit or Gradio

**Name:** Saravanan P  
**Roll No:** 24BAD105  
**Platform:** Google Colab  
**Pre-trained model:** `distilgpt2` (Hugging Face Transformers)  
**Interface:** Gradio



## Setup – Install required libraries


In [1]:
!pip -q install -U transformers gradio accelerate

## Student details and library imports

In [2]:
# EXPERIMENT 9 - Transformer-Based LLM Chatbot
# Name: Saravanan P
# Roll No: 24BAD105

import os
import time
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("Name: Saravanan P")
print("Roll No: 24BAD105")
print("Experiment 9: Transformer-Based LLM Chatbot")
print("PyTorch version:", torch.__version__)
print("Gradio version:", gr.__version__)
print("CUDA available:", torch.cuda.is_available())

Name: Saravanan P
Roll No: 24BAD105
Experiment 9: Transformer-Based LLM Chatbot
PyTorch version: 2.11.0+cpu
Gradio version: 6.31.0
CUDA available: False


# PART A – Loading and Configuring a Pre-trained LLM



In [3]:
# Select a pre-trained Transformer language model
model_name = "distilgpt2"

# Load tokenizer and model
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 tokenizers do not define a separate padding token by default.
# Use the end-of-sequence token for padding during generation.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Select GPU when available; otherwise use CPU
device = 0 if torch.cuda.is_available() else -1

# Create text-generation pipeline
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("Model loaded successfully:", model_name)

Loading tokenizer and model...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully: distilgpt2


### A1. Tokenize a sample prompt and inspect token IDs

In [4]:
# Sample user prompt
sample_prompt = "Explain artificial intelligence in simple words."

# Convert text into token IDs
encoded_prompt = tokenizer(sample_prompt, return_tensors="pt")

print("Original prompt:", sample_prompt)
print("Token IDs:", encoded_prompt["input_ids"].tolist())
print("Number of tokens:", encoded_prompt["input_ids"].shape[1])
print("Decoded tokens:", tokenizer.convert_ids_to_tokens(encoded_prompt["input_ids"][0]))

Original prompt: Explain artificial intelligence in simple words.
Token IDs: [[18438, 391, 11666, 4430, 287, 2829, 2456, 13]]
Number of tokens: 8
Decoded tokens: ['Expl', 'ain', 'Ġartificial', 'Ġintelligence', 'Ġin', 'Ġsimple', 'Ġwords', '.']


### A2. Generate a response with configured parameters

In [5]:
# Generation parameters
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 80

sample_result = generator(
    sample_prompt,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=True,
    temperature=TEMPERATURE,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
    num_return_sequences=1
)

print("Prompt:", sample_prompt)
print("Generated response:")
print(sample_result[0]["generated_text"][len(sample_prompt):].strip())

Prompt: Explain artificial intelligence in simple words.
Generated response:



# PART B – Implementing Chatbot Functionality

**Aim:** Create a function that accepts user queries, generates responses using the loaded LLM, and uses previous messages as context.

In [6]:
def build_chat_prompt(message, history=None, max_history_turns=4):
    """Create a simple text prompt with recent conversation history."""
    history = history or []
    prompt_parts = [
        "You are a helpful chatbot. Give clear and simple answers."
    ]

    # Support Gradio history in either message-dictionary or tuple format.
    for item in history[-max_history_turns:]:
        if isinstance(item, dict):
            role = item.get("role", "")
            content = item.get("content", "")
            if not isinstance(content, str):
                continue
            if role == "user":
                prompt_parts.append(f"User: {content}")
            elif role == "assistant":
                prompt_parts.append(f"Assistant: {content}")
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            user_text, assistant_text = item[0], item[1]
            if user_text:
                prompt_parts.append(f"User: {user_text}")
            if assistant_text:
                prompt_parts.append(f"Assistant: {assistant_text}")

    prompt_parts.append(f"User: {message}")
    prompt_parts.append("Assistant:")
    return "\n".join(prompt_parts)


def generate_chat_response(message, history=None, temperature=0.7, max_new_tokens=100):
    """Generate a chatbot response using the pre-trained language model."""
    if not isinstance(message, str) or not message.strip():
        return "Please enter a message."

    prompt = build_chat_prompt(message.strip(), history)
    result = generator(
        prompt,
        max_new_tokens=int(max_new_tokens),
        do_sample=True,
        temperature=max(float(temperature), 0.01),
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
        num_return_sequences=1
    )

    # Keep only newly generated text, not the prompt itself.
    full_text = result[0]["generated_text"]
    answer = full_text[len(prompt):].strip()

    # Remove accidental continuation into the next conversation turn.
    for stop_text in ["\nUser:", "\nHuman:"]:
        if stop_text in answer:
            answer = answer.split(stop_text, 1)[0].strip()

    return answer if answer else "I could not generate a clear response. Please try another prompt."


# Test chatbot with a sample query
user_query = "What is machine learning?"
chat_response = generate_chat_response(user_query, history=[])

print("User:", user_query)
print("Chatbot:", chat_response)

User: What is machine learning?
Chatbot: Machine learning allows us to identify the most important areas of our lives that matter, from making decisions on what we want to do with your business and whether you're going to go back or work for another company – something like this one in Seattle where I'm looking at how people think about myself."
The following words come up every day when someone says "I love me," so it's often useful to get started if they've had an opportunity to ask questions related to their job background. If there


### B1. Test the chatbot with different queries

In [7]:
test_queries = [
    "What is Python?",
    "Explain cloud computing in simple words.",
    "Why is data visualization useful?"
]

for query in test_queries:
    print("\nUser:", query)
    print("Chatbot:", generate_chat_response(query, history=[]))


User: What is Python?
Chatbot: The easiest way to use python, just make sure you have an active account with the following command line (in case I had it in your browser): "python" This means that any time someone tells me about how they can learn programming from others on their own without having to go through my training process or be told what program(s) they'd like them to implement - this will also lead to some very useful features for anyone using PyPy's software if necessary. It does require two lines of

User: Explain cloud computing in simple words.
Chatbot: Have you ever seen an unorganized group of people talk about this topic? What do they think the most important thing to know is that every person has been doing their best work for them, whether it be on Facebook or Skype? I believe all these things can help with business plans like getting rid of traffic congestion while your company does not want everyone else going crazy.

User: Why is data visualization useful?
Chatb

# PART C – Developing an Interactive Interface

**Aim:** Build an interactive chat-style interface using Gradio and connect it to the response-generation function.

The following cell launches a Gradio chat interface inside Google Colab. In Colab, `share=True` creates a temporary public link for testing. Do not enter passwords, private information, or confidential data into the demo.

In [11]:
# Create an interactive Gradio chatbot

def gradio_chat(message, history):
    return generate_chat_response(
        message,
        history=history,
        temperature=0.7,
        max_new_tokens=100
    )

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="Experiment 9 - Transformer LLM Chatbot",
    description="Ask a question to interact with the pre-trained distilgpt2 language model.",
    examples=[
        "What is artificial intelligence?",
        "Explain Python programming.",
        "What is the use of a database?"
    ]
)

# Launch the chat interface in Google Colab
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://16f4aa83e9e2ea46b8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://16f4aa83e9e2ea46b8.gradio.live


# PART D – Application Testing and Deployment

**Aim:** Test response quality and usability, observe limitations such as hallucinations and response latency, create project files, and prepare the project for GitHub / Hugging Face Spaces.

> The Experiment 9 manual includes two generic deployment tasks: organize the source code with `requirements.txt`, upload it to GitHub, and deploy it on a platform such as Hugging Face Spaces. The notebook creates the required project files below. Actual cloud deployment requires the student to sign in to the relevant service and upload the files.

### D1. Test response quality and measure response time

In [12]:
test_prompt = "Explain the difference between AI and machine learning."
start_time = time.time()
test_answer = generate_chat_response(test_prompt, history=[])
response_time = time.time() - start_time

print("Test prompt:", test_prompt)
print("Generated response:", test_answer)
print(f"Response time: {response_time:.2f} seconds")
print("\nObservation:")
print("Check whether the answer is relevant, understandable, and factually correct.")
print("A small pre-trained model may produce incomplete, repetitive, or incorrect responses.")

Test prompt: Explain the difference between AI and machine learning.
Generated response: How to develop automated programs that help you manage your data quickly.
Response time: 0.60 seconds

Observation:
Check whether the answer is relevant, understandable, and factually correct.
A small pre-trained model may produce incomplete, repetitive, or incorrect responses.


### D2. Observe chatbot limitations

In [13]:
limitations = [
    "Hallucination: the model may generate incorrect or misleading information.",
    "Limited context: only a short recent conversation history is included in the prompt.",
    "Response latency: generation may be slower on CPU or when the model is first loaded.",
    "Model capability: distilgpt2 is a compact language model and may not follow instructions reliably."
]

print("Observed / possible limitations:")
for number, limitation in enumerate(limitations, start=1):
    print(f"{number}. {limitation}")

Observed / possible limitations:
1. Hallucination: the model may generate incorrect or misleading information.
2. Limited context: only a short recent conversation history is included in the prompt.
3. Response latency: generation may be slower on CPU or when the model is first loaded.
4. Model capability: distilgpt2 is a compact language model and may not follow instructions reliably.


### D3. Create `requirements.txt`

In [14]:
requirements_text = """torch
transformers
gradio
accelerate
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements_text)

print("requirements.txt created successfully")
print(requirements_text)

requirements.txt created successfully
torch
transformers
gradio
accelerate



### D4. Create `app.py` for Hugging Face Spaces deployment

In [15]:
app_code = r'''import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = 0 if torch.cuda.is_available() else -1
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)


def build_prompt(message, history):
    parts = ["You are a helpful chatbot. Give clear and simple answers."]
    for item in (history or [])[-4:]:
        if isinstance(item, dict):
            role, content = item.get("role", ""), item.get("content", "")
            if isinstance(content, str) and role in ("user", "assistant"):
                parts.append(f"{role.capitalize()}: {content}")
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            if item[0]:
                parts.append(f"User: {item[0]}")
            if item[1]:
                parts.append(f"Assistant: {item[1]}")
    parts.extend([f"User: {message}", "Assistant:"])
    return "\n".join(parts)


def respond(message, history):
    if not message or not message.strip():
        return "Please enter a message."
    prompt = build_prompt(message.strip(), history)
    output = generator(
        prompt,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
        num_return_sequences=1
    )[0]["generated_text"]
    answer = output[len(prompt):].strip()
    for marker in ("\nUser:", "\nHuman:"):
        if marker in answer:
            answer = answer.split(marker, 1)[0].strip()
    return answer or "I could not generate a clear response. Please try again."


demo = gr.ChatInterface(
    fn=respond,
    title="Transformer-Based LLM Chatbot",
    description="A simple chatbot powered by the pre-trained distilgpt2 model.",
    examples=["What is artificial intelligence?", "Explain Python.", "What is a database?"]
)

demo.launch()
'''

with open("app.py", "w", encoding="utf-8") as file:
    file.write(app_code)

print("app.py created successfully")
print("Files available:", os.listdir("."))

app.py created successfully
Files available: ['.config', '.gradio', 'requirements.txt', 'app.py', 'sample_data']
